**Tabela** | ecommerce_produtos |

**Origem** | squad2/silver/ecommerce_produtos (Delta) |

**Destinos** | 
* `gold/ecommerce_produtos_kpi` (Fotografia Atual - Overwrite)
* `gold/ecommerce_produtos_historico` (Série Temporal - Append)
**Modo** | Delta Incremental (via Control JSON)
**Regras de Negócio Aplicadas** |
* Regra 4: Contagem de novos SKUs no lote (Alerta Carga de Teste se > 50)
* Regra 5: Alerta se produto ativo vier com preço zerado ou negativo

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
from deltalake import DeltaTable, write_deltalake
import pandas as pd
from datetime import datetime
import json

TABELA = "ecommerce_produtos"

# Caminhos ABFSS oficiais utilizando as variáveis globais do seu Helpers
path_silver    = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{TABELA}"
path_gold_kpi  = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/gold/{TABELA}_kpi"
path_gold_hist = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/gold/{TABELA}_historico"
path_control   = f"gold/control/{TABELA}.json"

print(f" Lendo de (Silver): {path_silver}")
print(f" Gravando em (Gold): {path_gold_kpi}")

try:
    # BLINDAGEM: Verifica se a tabela Silver já foi inicializada fisicamente
    if not DeltaTable.is_deltatable(path_silver, storage_options=get_storage_options()):
        print(f"✨ [Aviso Gold] A tabela Silver de {TABELA} ainda não foi inicializada.")
    else:
        # 1. Abre a tabela Silver com projeção de colunas para performance em memória
        dt_silver = DeltaTable(path_silver, storage_options=get_storage_options())
        df_pandas = dt_silver.to_pandas(columns=['sku', 'is_ativo', 'preco_lista', 'bronze_source_file'])
        
        # 2. CONTROLE INCREMENTAL: Instancia o cliente de controle usando o Azure SDK nativo
        squad2_client = get_squad2_client()
        file_client = squad2_client.get_file_client(path_control)
        
        processados = set()
        if file_client.exists():
            conteudo = file_client.download_file().readall().decode('utf-8')
            processados = set(json.loads(conteudo))
        
        # Filtra apenas os arquivos inéditos trazidos pela Silver
        df_novos_dados = df_pandas[~df_pandas['bronze_source_file'].isin(processados)].copy()
        
        if df_novos_dados.empty:
            print(" Camada Gold de Produtos em dia! Nenhum dado novo para processar.")
        else:
            print(f" Processando {len(df_novos_dados)} novas linhas da Silver...")
            
            # 3. APLICAÇÃO DAS REGRAS DE NEGÓCIO DA PLANILHA (PRODUTOS)
            # Regra 4: Contagem de SKUs únicos e verificação de carga de teste
            novos_skus_count = df_novos_dados['sku'].nunique()
            alerta_carga_teste = "SIM" if novos_skus_count > 50 else "NAO"
            
            # Regra 5: Alerta se produto ativo vier com preço zerado ou negativo
            df_preco_bug = df_novos_dados[(df_novos_dados['is_ativo'] == True) & (df_novos_dados['preco_lista'] <= 0)]
            alerta_bug_preco = "SIM" if not df_preco_bug.empty else "NAO"
            
            # Montagem estruturada do DataFrame analítico final
            df_pandas_kpi = pd.DataFrame([{
                "data_analise": str(datetime.now().date()),
                "horario_analise": datetime.now().time().strftime("%H:%M:%S"),
                "novos_skus_no_lote": int(novos_skus_count),
                "alerta_carga_teste_gt50": alerta_carga_teste,
                "alerta_produto_ativo_preco_zero": alerta_bug_preco
            }])
            
            # 4. COLUNA DE AUDITORIA DA GOLD
            df_pandas_kpi['gold_processed_at'] = datetime.now()
            
            # Remove os fuso-horários para o formato Delta
            for col in df_pandas_kpi.columns:
                if pd.api.types.is_datetime64_any_dtype(df_pandas_kpi[col]):
                    df_pandas_kpi[col] = df_pandas_kpi[col].dt.tz_localize(None)
                    
            # -------------------------------------------------------------------------
            # 5. SINK 1: GRAVAÇÃO NO LAKEHOUSE DELTA (Duplo Sink)
            # -------------------------------------------------------------------------
            # A) Visão de KPIs rápidos (Overwrite)
            write_deltalake(path_gold_kpi, df_pandas_kpi, mode="overwrite", storage_options=get_storage_options())
            
            # B) Visão de Linha do Tempo Histórica (Append)
            write_deltalake(path_gold_hist, df_pandas_kpi, mode="append", storage_options=get_storage_options())
            
            # -------------------------------------------------------------------------
            # 6. SINK 2: INGESTÃO NO SQL SERVER VIA PROVEDOR OFICIAL DO SQUAD
            # -------------------------------------------------------------------------
            # Converte as métricas calculadas em DataFrame Spark para usar a infraestrutura interna
            spark_df = spark.createDataFrame(df_pandas_kpi)
            
            # Grava a tabela de KPIs atuais (Modo Overwrite)
            spark_df.write \
                .format("sqlserver") \
                .options(**SQL_OPTIONS) \
                .option("dbtable", f"{TABELA}_kpi") \
                .mode("overwrite") \
                .save()
            
            # Grava a tabela histórica acumulada (Modo Append)
            spark_df.write \
                .format("sqlserver") \
                .options(**SQL_OPTIONS) \
                .option("dbtable", f"{TABELA}_historico") \
                .mode("append") \
                .save()
            
            # 7. ATUALIZA O CONTROL JSON NA CAMADA GOLD
            arquivos_atuais = set(df_novos_dados['bronze_source_file'].unique())
            todos_processados = list(processados.union(arquivos_atuais))
            file_client.upload_data(json.dumps(todos_processados), overwrite=True)
            
            print("\n SUCESSO ABSOLUTO! Pipeline integrado de ponta a ponta seguindo as diretrizes do Squad 2!")
            display(df_pandas_kpi)

except Exception as e:
    print(f" Erro no processamento: {str(e)}")
    raise